# 2. Wave power resource

Converts the observed records into wave power flux and compares the three
candidate sites.

## What changed from the original version of this notebook

The previous notebook computed `Pflux = 0.49 * H**2 * T` using `VTM02`, and the
library module used the same physics with `VTPK`. **Both were wrong**, in
opposite directions.

The deep-water energy-flux formula

$$P = \frac{\rho g^2}{64\pi} H_{m0}^2 T_e$$

is defined on the **energy period** $T_e = m_{-1}/m_0$ — not the zero-crossing
period and not the peak period. Using $T_p$ raw overstates flux by roughly 16%;
using $T_{m02}$ raw understates it by roughly 17%. Any revenue figure computed
the old way inherits that error.

`src/processing/wave_power_flux.py` now prefers the correct variable and applies
documented JONSWAP conversions when only an approximation is available, and
records which one it used in the output attributes. Three further fixes: the
"worst site" was previously the worst site *above the viability threshold*
rather than the worst site; monthly aggregation used a deprecated pandas alias;
and the revenue calculation hard-coded device parameters that now live in
`DeviceSpec`.

In [ ]:
# Run from anywhere in the repo: put the project root on the path.
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 5)
print(f"project root: {root}")

In [ ]:
from src.config import NDBC_STATIONS, PILOT_STATIONS
from src.data.ndbc import load_station
from src.processing.wave_power_flux import (
    ENERGY_PERIOD_FACTORS,
    calculate_consistency_metrics,
    calculate_wave_power_flux,
)

YEARS = list(range(2015, 2024))
records = {s: load_station(s, YEARS) for s in PILOT_STATIONS}
ENERGY_PERIOD_FACTORS

## Period conversion

NDBC publishes `APD` (average period, close to the zero-crossing $T_{m02}$) and
`DPD` (dominant/peak period, $T_p$). Neither is $T_e$ directly, so a documented
spectral conversion is applied. The factor used is shown so the assumption is
visible rather than buried.

The size of the correction is worth seeing explicitly — it is not a rounding
detail.

In [ ]:
def power_flux_series(df):
    """Wave power flux in W/m from an NDBC record, using the energy period."""
    energy_period = df["APD"] * ENERGY_PERIOD_FACTORS["VTM02"]
    return calculate_wave_power_flux(df["WVHT"], energy_period)

comparison = {}
for station_id, df in records.items():
    correct = power_flux_series(df).mean() / 1000.0
    naive_tm02 = calculate_wave_power_flux(df["WVHT"], df["APD"]).mean() / 1000.0
    naive_tp = calculate_wave_power_flux(df["WVHT"], df["DPD"]).mean() / 1000.0
    comparison[station_id] = {
        "correct_Te_kW_per_m": correct,
        "naive_APD_kW_per_m": naive_tm02,
        "naive_DPD_kW_per_m": naive_tp,
        "error_if_APD_used_%": 100 * (naive_tm02 / correct - 1),
        "error_if_DPD_used_%": 100 * (naive_tp / correct - 1),
    }
pd.DataFrame(comparison).T.round(2)

## Resource by site

In [ ]:
flux = {s: power_flux_series(df) for s, df in records.items()}

resource = {}
for station_id, series in flux.items():
    metrics = calculate_consistency_metrics(series.dropna().to_numpy())
    resource[station_id] = {
        "name": NDBC_STATIONS[station_id].name,
        "mean_kW_per_m": metrics["mean_power"] / 1000.0,
        "median_kW_per_m": metrics["median_power"] / 1000.0,
        "cv": metrics["coefficient_of_variation"],
        "pct_above_p75": metrics["percent_above_threshold"],
        "winter_kW_per_m": series[series.index.month.isin([12, 1, 2])].mean() / 1000.0,
        "summer_kW_per_m": series[series.index.month.isin([6, 7, 8])].mean() / 1000.0,
    }
resource_df = pd.DataFrame(resource).T
resource_df

In [ ]:
# Winter/summer ratio: the seasonality that drives both revenue and access.
resource_df["seasonal_ratio"] = (
    resource_df["winter_kW_per_m"] / resource_df["summer_kW_per_m"]
)
resource_df = resource_df.astype(
    {"mean_kW_per_m": float, "median_kW_per_m": float, "seasonal_ratio": float}
)
resource_df.round(2).sort_values("mean_kW_per_m", ascending=False)

Note how far the **mean** sits above the **median** at every site. Wave power
goes as $H^2$, so the distribution is strongly right-skewed: a small number of
storm hours carry a large share of the annual energy. A device that shuts down
in those hours loses more than its share of them — which is why notebook 3
applies a survival cut-out rather than integrating the raw flux.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for station_id, series in flux.items():
    monthly = (series / 1000.0).groupby(series.index.month).mean()
    axes[0].plot(monthly.index, monthly.values, marker="o", label=station_id)
axes[0].set_xlabel("month"); axes[0].set_ylabel("mean flux (kW/m)")
axes[0].set_title("Seasonal wave power"); axes[0].grid(alpha=0.3); axes[0].legend()

for station_id, series in flux.items():
    axes[1].hist((series / 1000.0).dropna(), bins=80, range=(0, 200),
                 histtype="step", density=True, label=station_id)
axes[1].set_xlabel("flux (kW/m)"); axes[1].set_ylabel("density")
axes[1].set_title("Distribution: a long right tail carries the energy")
axes[1].set_yscale("log"); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# What share of annual energy arrives in the top 10% of hours?
for station_id, series in flux.items():
    clean = series.dropna().sort_values(ascending=False)
    share = clean.iloc[: int(0.1 * len(clean))].sum() / clean.sum()
    print(f"{station_id}: top 10% of hours carry {100 * share:.0f}% of the energy")

## Ranking on resource alone

This is the answer a conventional resource assessment gives, and it is the
answer the original notebook gave. Notebook 3 shows why it is not the answer to
act on: it takes no account of whether the site can be reached, and access cost
rises with exactly the quantity being ranked.

In [ ]:
resource_df[["name", "mean_kW_per_m", "cv", "seasonal_ratio"]].sort_values(
    "mean_kW_per_m", ascending=False
).round(2)